In [ ]:
#Question 1: Custom Feature Transformation Using FunctionTransformer
#A. Use FunctionTransformer with np.log1p to transform both numerical features. Compare the original and transformed means, ranges, and standard deviations. Explain mathematically why the transformation compresses large values more strongly than small values.
import numpy as np
import pandas as pd
from sklearn.preprocessing import FunctionTransformer

# Create the dataset

df = pd.DataFrame({
"Customer_ID": range(1, 11),
"Income": [15000, 18000, 22000, 25000, 30000,
35000, 42000, 55000, 120000, 500000],
"Monthly_Expenses": [8000, 9000, 11000, 12000, 15000,
17000, 20000, 25000, 45000, 180000]
})

# Create the logarithmic transformer

log_transformer = FunctionTransformer(
np.log1p,
feature_names_out="one-to-one"
)

# Transform the two numerical features

features = ["Income", "Monthly_Expenses"]

transformed_data = log_transformer.fit_transform(df[features])

# Create a DataFrame for transformed values using the underlying numpy array

df_log = pd.DataFrame(
    transformed_data.values,
    columns=["Income_Log", "Expenses_Log"]
)

print("Original Data:")
print(df[features])

print("\nLog-Transformed Data:")
print(df_log.round(3))

# Compare statistical measures

for column, log_column in zip(
    features,
    ["Income_Log", "Expenses_Log"]
):
    original = df[column]
    transformed = df_log[log_column]

    print(f"\nStatistics for {column}")
    print("Original Mean:", round(original.mean(), 3))
    print("Transformed Mean:", round(transformed.mean(), 3))
    print("Original Range:", round(original.max() - original.min(), 3))
    print("Transformed Range:", round(transformed.max() - transformed.min(), 3))
    print("Original Standard Deviation:", round(original.std(), 3))
    print("Transformed Standard Deviation:", round(transformed.std(), 3))

Original Data:
   Income  Monthly_Expenses
0   15000              8000
1   18000              9000
2   22000             11000
3   25000             12000
4   30000             15000
5   35000             17000
6   42000             20000
7   55000             25000
8  120000             45000
9  500000            180000

Log-Transformed Data:
   Income_Log  Expenses_Log
0       9.616         8.987
1       9.798         9.105
2       9.999         9.306
3      10.127         9.393
4      10.309         9.616
5      10.463         9.741
6      10.645         9.904
7      10.915        10.127
8      11.695        10.714
9      13.122        12.101

Statistics for Income
Original Mean: 86200.0
Transformed Mean: 10.669
Original Range: 485000
Transformed Range: 3.506
Original Standard Deviation: 148573.813
Transformed Standard Deviation: 1.05

Statistics for Monthly_Expenses
Original Mean: 34200.0
Transformed Mean: 9.899
Original Range: 172000
Transformed Range: 3.113
Original Standard Devi

In [ ]:
#B. Create a scikit-learn pipeline that applies the logarithmic transformation before fitting a LinearRegression model to predict monthly expenses from income. Compare its performance against a model trained on untransformed income using a train-test split and an appropriate regression metric. Interpret the results without assuming the transformation must perform better.
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# Create the dataset

df = pd.DataFrame({
"Customer_ID": range(1, 11),
"Income": [15000, 18000, 22000, 25000, 30000,
35000, 42000, 55000, 120000, 500000],
"Monthly_Expenses": [8000, 9000, 11000, 12000, 15000,
17000, 20000, 25000, 45000, 180000]
})

# Input and target

X = df [["Income"]]
y = df["Monthly_Expenses"]

# Split the data

X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.3, random_state=42
)

# Model 1: Original income

model_original = LinearRegression()
model_original.fit(X_train, y_train)

pred_original = model_original.predict(X_test)

# Model 2: Log-transformed income

model_log = Pipeline([
("log_transform", FunctionTransformer(
np.log1p,
feature_names_out="one-to-one"
)),
("regression", LinearRegression())
])

model_log.fit(X_train, y_train)
pred_log = model_log.predict(X_test)

# Evaluate both models

results = pd.DataFrame({
"Model": ["Original Income", "Log-Transformed Income"],
"MAE": [
mean_absolute_error(y_test, pred_original),
mean_absolute_error(y_test, pred_log)
],
"RMSE": [
root_mean_squared_error(y_test, pred_original),
root_mean_squared_error(y_test, pred_log)
]
})

print(results.round(2))

                    Model       MAE      RMSE
0         Original Income   1108.23   1158.97
1  Log-Transformed Income  24318.31  29015.14


In [ ]:
#2 Designing and Validating Custom Transformations
#A. Use FunctionTransformer to create and apply a custom function that converts sales into thousands of units. Apply a separate square-root transformation to Transaction_Count. Compare the transformed values with the original values and explain when each transformation could be useful
An e-commerce company wants to transform sales and transaction features before training a machine learning mode
import numpy as np
import pandas as pd
from sklearn.preprocessing import FunctionTransformer

# Create the dataset

df = pd.DataFrame({
"Sales": [100, 150, 200, 250, 400, 600, 900, 1500, 3000, 8000],
"Transaction_Count": [1, 2, 2, 3, 4, 5, 7, 10, 15, 30],
"Discount_Percent": [0, 5, 10, 15, 20, 5, 10, 25, 30, 40],
"Revenue": [120, 170, 230, 280, 450, 650, 950, 1600, 3200, 8500]
})

# Custom function to convert sales into thousands

def convert_to_thousands(X):
    return np.asarray(X) / 1000

sales_transformer = FunctionTransformer(
convert_to_thousands,
feature_names_out="one-to-one"
)

# Square root transformer

sqrt_transformer = FunctionTransformer(
np.sqrt,
feature_names_out="one-to-one"
)

# Apply transformations (using .ravel() directly because sales_transformer returns a numpy array)

df["Sales_Thousands"] = sales_transformer.fit_transform(
df[["Sales"]]
).ravel()

# Since sqrt_transformer might return a DataFrame, we make sure to convert or access appropriately
transformed_sqrt = sqrt_transformer.fit_transform(df[["Transaction_Count"]])
df["Transaction_Count_Sqrt"] = (
    transformed_sqrt.values.ravel() if hasattr(transformed_sqrt, "values") else transformed_sqrt.ravel()
)

# Compare original and transformed columns

print(df[[
"Sales",
"Sales_Thousands",
"Transaction_Count",
"Transaction_Count_Sqrt"
]].round(3))

   Sales  Sales_Thousands  Transaction_Count  Transaction_Count_Sqrt
0    100             0.10                  1                   1.000
1    150             0.15                  2                   1.414
2    200             0.20                  2                   1.414
3    250             0.25                  3                   1.732
4    400             0.40                  4                   2.000
5    600             0.60                  5                   2.236
6    900             0.90                  7                   2.646
7   1500             1.50                 10                   3.162
8   3000             3.00                 15                   3.873
9   8000             8.00                 30                   5.477


In [ ]:
#B. Implement a reusable custom transformation that clips a numerical feature to a specified lower and upper bound. Apply it to Sales, compare the results for at least two different clipping ranges, and demonstrate how clipping changes the mean,
# standard deviation, and extreme values. Explain why clipping should be based on domain knowledge rather than arbitrary limits
import numpy as np
import pandas as pd
from sklearn.preprocessing import FunctionTransformer

df = pd.DataFrame({
"Sales": [100, 150, 200, 250, 400, 600, 900, 1500, 3000, 8000]
})

# Reusable clipping function

def clip_values(X, lower, upper):
    return np.clip(np.asarray(X), lower, upper)

# Define two different clipping ranges

clip_1 = FunctionTransformer(
clip_values,
kw_args={"lower": 200, "upper": 3000},
feature_names_out="one-to-one"
)

clip_2 = FunctionTransformer(
clip_values,
kw_args={"lower": 100, "upper": 1500},
feature_names_out="one-to-one"
)

# Apply transformations

df["Clipped_200_3000"] = clip_1.fit_transform(
df[["Sales"]]
).ravel()

df["Clipped_100_1500"] = clip_2.fit_transform(
df[["Sales"]]
).ravel()

print("Original and Clipped Values:")
print(df)

# Compare statistics

for column in ["Sales", "Clipped_200_3000", "Clipped_100_1500"]:
    print(f"\nStatistics for {column}")
    print("Mean:", round(df[column].mean(), 2))
    print("Standard Deviation:", round(df[column].std(), 2))
    print("Minimum:", df[column].min())
    print("Maximum:", df[column].max())
    print("Range:", df[column].max() - df[column].min())

Original and Clipped Values:
   Sales  Clipped_200_3000  Clipped_100_1500
0    100               200               100
1    150               200               150
2    200               200               200
3    250               250               250
4    400               400               400
5    600               600               600
6    900               900               900
7   1500              1500              1500
8   3000              3000              1500
9   8000              3000              1500

Statistics for Sales
Mean: 1510.0
Standard Deviation: 2447.54
Minimum: 100
Maximum: 8000
Range: 7900

Statistics for Clipped_200_3000
Mean: 1025.0
Standard Deviation: 1118.34
Minimum: 200
Maximum: 3000
Range: 2800

Statistics for Clipped_100_1500
Mean: 710.0
Standard Deviation: 593.86
Minimum: 100
Maximum: 1500
Range: 1400


In [ ]:
#Question 3: PowerTransformer and Distribution Analysis
#A. Apply PowerTransformer(method="yeo-johnson") to all three numerical features. Calculate and compare their skewness before and after transformation. Display the learned lambda values and explain why transformed data is not guaranteed to be perfectly normally distributed
import numpy as np
import pandas as pd
from sklearn.preprocessing import PowerTransformer

# Create the dataset

df = pd.DataFrame({
"Annual_Income": [20000, 22000, 25000, 28000, 32000,
40000, 50000, 70000, 150000, 600000],
"Loan_Amount": [5000, 7000, 9000, 12000, 15000,
20000, 30000, 50000, 100000, 400000],
"Account_Balance": [-500, 0, 200, 500, 1000,
2500, 4000, 8000, 15000, 50000]
})

# Create Yeo-Johnson transformer

pt = PowerTransformer(method="yeo-johnson")

# Fit and transform the data

transformed_values = pt.fit_transform(df)

df_transformed = pd.DataFrame(
transformed_values,
columns=df.columns,
index=df.index
)

# Compare skewness before and after

skewness_comparison = pd.DataFrame({
"Original_Skewness": df.skew(),
"Transformed_Skewness": df_transformed.skew()
})

print("Original Data:")
print(df)

print("\nTransformed Data:")
print(df_transformed.round(3))

print("\nSkewness Comparison:")
print(skewness_comparison.round(3))

# Display learned lambda values

lambda_values = pd.Series(
pt.lambdas_,
index=df.columns,
name="Learned_Lambda"
)

print("\nLearned Lambda Values:")
print(lambda_values.round(4))


Original Data:
   Annual_Income  Loan_Amount  Account_Balance
0          20000         5000             -500
1          22000         7000                0
2          25000         9000              200
3          28000        12000              500
4          32000        15000             1000
5          40000        20000             2500
6          50000        30000             4000
7          70000        50000             8000
8         150000       100000            15000
9         600000       400000            50000

Transformed Data:
   Annual_Income  Loan_Amount  Account_Balance
0         -1.380       -1.599           -1.791
1         -1.135       -1.119           -0.435
2         -0.834       -0.796           -0.389
3         -0.591       -0.460           -0.344
4         -0.332       -0.223           -0.282
5          0.046        0.057           -0.131
6          0.364        0.406           -0.004
7          0.749        0.780            0.286
8          1.323        1.

In [ ]:
#B. Apply PowerTransformer(method="box-cox") separately to the strictly positive Annual_Income and Loan_Amount features. Then attempt to apply Box-Cox to Account_Balance. Explain the validity of each operation and implement a safe preprocessing approach that selects a suitable transformation based on whether a feature contains zero or negative values.
import numpy as np
import pandas as pd

from sklearn.preprocessing import PowerTransformer
from sklearn.compose import ColumnTransformer

# Create the dataset

df = pd.DataFrame({
"Annual_Income": [20000, 22000, 25000, 28000, 32000,
40000, 50000, 70000, 150000, 600000],
"Loan_Amount": [5000, 7000, 9000, 12000, 15000,
20000, 30000, 50000, 100000, 400000],
"Account_Balance": [-500, 0, 200, 500, 1000,
2500, 4000, 8000, 15000, 50000]
})

# Box-Cox for Annual_Income

income_boxcox = PowerTransformer(method="box-cox")
income_transformed = income_boxcox.fit_transform(
df[["Annual_Income"]]
)

print("Box-Cox Annual Income:")
print(income_transformed.round(3))

# Box-Cox for Loan_Amount

loan_boxcox = PowerTransformer(method="box-cox")
loan_transformed = loan_boxcox.fit_transform(
df[["Loan_Amount"]]
)

print("\nBox-Cox Loan Amount:")
print(loan_transformed.round(3))

# Attempt Box-Cox on Account_Balance

try:
    balance_boxcox = PowerTransformer(method="box-cox")
    balance_boxcox.fit_transform(df[["Account_Balance"]])
except ValueError as error:
    print("\nBox-Cox failed for Account_Balance:")
    print(error)

# Safe approach:

# Use Box-Cox for strictly positive features.

# Use Yeo-Johnson for a feature containing zero or negative values.

preprocessor = ColumnTransformer(
transformers=[
(
"income_boxcox",
PowerTransformer(method="box-cox"),
["Annual_Income"]
),
(
"loan_boxcox",
PowerTransformer(method="box-cox"),
["Loan_Amount"]
),
(
"balance_yeojohnson",
PowerTransformer(method="yeo-johnson"),
["Account_Balance"]
)
],
verbose_feature_names_out=False
)

safe_transformed = preprocessor.fit_transform(df)

safe_df = pd.DataFrame(
safe_transformed,
columns=preprocessor.get_feature_names_out()
)

print("\nSafely Transformed Data:")
print(safe_df.round(3))

Box-Cox Annual Income:
[[-1.38 ]
 [-1.135]
 [-0.834]
 [-0.591]
 [-0.332]
 [ 0.046]
 [ 0.364]
 [ 0.749]
 [ 1.323]
 [ 1.79 ]]

Box-Cox Loan Amount:
[[-1.599]
 [-1.119]
 [-0.796]
 [-0.46 ]
 [-0.223]
 [ 0.057]
 [ 0.406]
 [ 0.78 ]
 [ 1.19 ]
 [ 1.762]]

Box-Cox failed for Account_Balance:
The Box-Cox transformation can only be applied to strictly positive data

Safely Transformed Data:
   Annual_Income  Loan_Amount  Account_Balance
0         -1.380       -1.599           -1.791
1         -1.135       -1.119           -0.435
2         -0.834       -0.796           -0.389
3         -0.591       -0.460           -0.344
4         -0.332       -0.223           -0.282
5          0.046        0.057           -0.131
6          0.364        0.406           -0.004
7          0.749        0.780            0.286
8          1.323        1.190            0.715
9          1.790        1.762            2.376
